In [ ]:
!pip install -q transformers==4.43.3

In [ ]:
%cd /content
!git clone -b totoro3 https://github.com/camenduru/ComfyUI /content/TotoroUI
!pip install -q torch torchvision torchaudio torchsde einops diffusers accelerate xformers==0.0.28.post2
!apt -y install -qq aria2

In [ ]:
import os

# 1. Setup Environment (Will quickly skip if already done)
%cd /content
!git clone -b totoro3 https://github.com/camenduru/ComfyUI /content/TotoroUI || echo "Already cloned"
%cd /content/TotoroUI
!mkdir -p /content/TotoroUI/models/unet
!mkdir -p /content/TotoroUI/models/vae
!mkdir -p /content/TotoroUI/models/clip

# Clear out any broken gated downloads
!rm -f /content/TotoroUI/models/unet/*.aria2
!rm -f /content/TotoroUI/models/unet/flux1-schnell.safetensors

# 2. Download Models (Ungated FP8 mirror)
print("Downloading UNGATED FP8 UNET (approx 11.9GB)...")
!aria2c -c -x 16 -s 16 -k 1M https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-schnell-fp8-e4m3fn.safetensors -d /content/TotoroUI/models/unet -o flux1-schnell.safetensors

print("Verifying VAE and CLIP models... (Will skip if already downloaded)")
!aria2c -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.sft -d /content/TotoroUI/models/vae -o ae.sft
!aria2c -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/clip_l.safetensors -d /content/TotoroUI/models/clip -o clip_l.safetensors
!aria2c -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/t5xxl_fp8_e4m3fn.safetensors -d /content/TotoroUI/models/clip -o t5xxl_fp8_e4m3fn.safetensors

# --- VERIFY FILES EXIST ---
unet_path = "/content/TotoroUI/models/unet/flux1-schnell.safetensors"
if not os.path.exists(unet_path):
    raise FileNotFoundError("Download failed! The URL might be blocked by your network.")
else:
    print("Downloads verified successfully! Booting up FLUX...")

# 3. Imports and Initialization
import random
import torch
import numpy as np
from PIL import Image
import nodes
from nodes import NODE_CLASS_MAPPINGS
from totoro_extras import nodes_custom_sampler
import folder_paths

# --- THE CACHE FIX ---
original_get_full_path = folder_paths.get_full_path

def custom_get_full_path(folder_name, filename):
    if filename == "flux1-schnell.safetensors": return "/content/TotoroUI/models/unet/flux1-schnell.safetensors"
    if filename == "ae.sft": return "/content/TotoroUI/models/vae/ae.sft"
    if filename == "clip_l.safetensors": return "/content/TotoroUI/models/clip/clip_l.safetensors"
    if filename == "t5xxl_fp8_e4m3fn.safetensors": return "/content/TotoroUI/models/clip/t5xxl_fp8_e4m3fn.safetensors"
    return original_get_full_path(folder_name, filename)

folder_paths.get_full_path = custom_get_full_path

# 4. Initialize Nodes  ← THIS IS WHERE THE BUG WAS (truncated line)
DualCLIPLoader = NODE_CLASS_MAPPINGS["DualCLIPLoader"]()
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
RandomNoise = nodes_custom_sampler.NODE_CLASS_MAPPINGS["RandomNoise"]()
BasicGuider = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicGuider"]()
KSamplerSelect = nodes_custom_sampler.NODE_CLASS_MAPPINGS["KSamplerSelect"]()
BasicScheduler = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicScheduler"]()
SamplerCustomAdvanced = nodes_custom_sampler.NODE_CLASS_MAPPINGS["SamplerCustomAdvanced"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

# 5. Load Weights into Memory
with torch.inference_mode():
    clip = DualCLIPLoader.load_clip("t5xxl_fp8_e4m3fn.safetensors", "clip_l.safetensors", "flux")[0]
    unet = UNETLoader.load_unet("flux1-schnell.safetensors", "fp8_e4m3fn")[0]
    vae = VAELoader.load_vae("ae.sft")[0]

# Helper Function
def closestNumber(n, m):
    q = int(n / m)
    n1 = m * q
    if (n * m) > 0:
        n2 = m * (q + 1)
    else:
        n2 = m * (q - 1)
    if abs(n - n1) < abs(n - n2):
        return n1
    return n2

print("Setup Complete! Models loaded successfully.")

In [ ]:
import time
from totoro import model_management  # ← add this

with torch.inference_mode():
    positive_prompt = "A logo made of solid shapes that is titled as (Clapsticks)"
    width = 480
    height = 720
    seed = 0
    steps = 4
    sampler_name = "euler"
    scheduler = "simple"

    if seed == 0:
        seed = random.randint(0, 18446744073709551615)
    print(seed)

    start_time = time.time()

    cond, pooled = clip.encode_from_tokens(clip.tokenize(positive_prompt), return_pooled=True)
    cond = [[cond, {"pooled_output": pooled}]]
    noise = RandomNoise.get_noise(seed)[0]
    guider = BasicGuider.get_guider(unet, cond)[0]
    sampler = KSamplerSelect.get_sampler(sampler_name)[0]
    sigmas = BasicScheduler.get_sigmas(unet, scheduler, steps, 1.0)[0]
    latent_image = EmptyLatentImage.generate(closestNumber(width, 16), closestNumber(height, 16))[0]
    sample, sample_denoised = SamplerCustomAdvanced.sample(noise, guider, sampler, sigmas, latent_image)
    model_management.soft_empty_cache()
    decoded = VAEDecode.decode(vae, sample)[0].detach()
    Image.fromarray(np.array(decoded*255, dtype=np.uint8)[0]).save("/content/flux.png")

    elapsed = time.time() - start_time
    if elapsed >= 60:
        print(f"Generated in {elapsed / 60:.1f} minutes")
    else:
        print(f"Generated in {elapsed:.1f} seconds")

Image.fromarray(np.array(decoded*255, dtype=np.uint8)[0])